In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
import pandas as pd
import time
from matplotlib import pyplot as plt
%config InlineBackend.figure_format = 'retina'

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

In [ ]:
train = pd.read_csv('/content/drive/My Drive/HTC_CNN_dataset/WOS/wos_test_final.csv')
test = pd.read_csv('/content/drive/My Drive/HTC_CNN_dataset/WOS/wos_test_final.csv')

In [ ]:
data = train

texts = data['text'].values
labels_l1 = data['l1'].values
labels_l2 = data['l2'].values

label_encoder_l1 = LabelEncoder()
label_encoder_l2 = LabelEncoder()

encoded_l1 = label_encoder_l1.fit_transform(labels_l1)
encoded_l2 = label_encoder_l2.fit_transform(labels_l2)

tokenizer = Tokenizer(num_words=1800)
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
X_padded = pad_sequences(sequences, maxlen=180)

y_l1 = to_categorical(encoded_l1)
y_l2 = to_categorical(encoded_l2)

num_classes_l1 = len(label_encoder_l1.classes_)
num_classes_l2 = len(label_encoder_l2.classes_)

In [ ]:
early_stopping = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)

In [ ]:
def build_cnn_model(num_classes):
    input_text = Input(shape=(180,))
    embedding_text = Embedding(input_dim=1800, output_dim=64, input_length=180)(input_text)
    conv_text = Conv1D(filters=64, kernel_size=3, activation='relu')(embedding_text)
    global_pool_text = GlobalMaxPooling1D()(conv_text)
    dense_text = Dense(64, activation='relu')(global_pool_text)
    output = Dense(num_classes, activation='softmax')(dense_text)
    model = Model(inputs=input_text, outputs=output)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
model_l1 = build_cnn_model(num_classes_l1)
start_l1 = time.time()
history_l1 = model_l1.fit(X_padded, y_l1, epochs=10, batch_size=128, validation_split=0.2, callbacks=[early_stopping, reduce_lr])
end_l1 = time.time()

time_l1 = round(end_l1 - start_l1)

Epoch 1/10
98/98 [==============================] - 9s 73ms/step - loss: 1.6412 - accuracy: 0.3747 - val_loss: 1.4559 - val_accuracy: 0.4190 - lr: 0.0010
Epoch 2/10
98/98 [==============================] - 6s 64ms/step - loss: 0.9376 - accuracy: 0.6726 - val_loss: 0.7897 - val_accuracy: 0.7413 - lr: 0.0010
Epoch 3/10
98/98 [==============================] - 8s 78ms/step - loss: 0.5738 - accuracy: 0.8039 - val_loss: 0.5765 - val_accuracy: 0.7889 - lr: 0.0010
Epoch 4/10
98/98 [==============================] - 6s 60ms/step - loss: 0.4149 - accuracy: 0.8617 - val_loss: 0.5059 - val_accuracy: 0.8173 - lr: 0.0010
Epoch 5/10
98/98 [==============================] - 8s 83ms/step - loss: 0.3151 - accuracy: 0.8952 - val_loss: 0.4857 - val_accuracy: 0.8285 - lr: 0.0010
Epoch 6/10
98/98 [==============================] - 6s 59ms/step - loss: 0.2346 - accuracy: 0.9277 - val_loss: 0.5004 - val_accuracy: 0.8266 - lr: 0.0010
Epoch 7/10
98/98 [==============================] - 7s 73ms/step - loss: 0.1

In [ ]:
pred_l1 = model_l1.predict(X_padded)
pred_labels_l1 = label_encoder_l1.inverse_transform(np.argmax(pred_l1, axis=1))
accuracy_l1 = np.mean(pred_labels_l1 == label_encoder_l1.inverse_transform(np.argmax(y_l1, axis=1)))

print(f'训练集 l1 的准确率: {accuracy_l1:.4f}')
print(f'Model_l1 time:{time_l1}')

490/490 [==============================] - 3s 6ms/step
训练集 l1 的准确率: 0.9118
Model_l1 time:50


In [ ]:
model_l2 = build_cnn_model(num_classes_l2)
start_l2 = time.time()
history_l2 = model_l2.fit(X_padded, y_l2, epochs=10, batch_size=128, validation_split=0.2, callbacks=[early_stopping, reduce_lr])
end_l2 = time.time()
time_l2 = round(end_l2 - start_l2)

Epoch 1/10
98/98 [==============================] - 7s 62ms/step - loss: 4.8507 - accuracy: 0.0192 - val_loss: 4.7733 - val_accuracy: 0.0073 - lr: 0.0010
Epoch 2/10
98/98 [==============================] - 8s 83ms/step - loss: 4.3591 - accuracy: 0.1179 - val_loss: 3.7989 - val_accuracy: 0.2664 - lr: 0.0010
Epoch 3/10
98/98 [==============================] - 6s 56ms/step - loss: 3.0916 - accuracy: 0.3704 - val_loss: 2.5645 - val_accuracy: 0.4532 - lr: 0.0010
Epoch 4/10
98/98 [==============================] - 8s 82ms/step - loss: 2.2189 - accuracy: 0.5125 - val_loss: 2.1272 - val_accuracy: 0.5442 - lr: 0.0010
Epoch 5/10
98/98 [==============================] - 7s 68ms/step - loss: 1.8191 - accuracy: 0.5847 - val_loss: 1.9381 - val_accuracy: 0.5759 - lr: 0.0010
Epoch 6/10
98/98 [==============================] - 9s 89ms/step - loss: 1.5750 - accuracy: 0.6255 - val_loss: 1.8645 - val_accuracy: 0.5902 - lr: 0.0010
Epoch 7/10
98/98 [==============================] - 6s 58ms/step - loss: 1.3

In [ ]:
pred_l2 = model_l2.predict(X_padded)
pred_labels_l2 = label_encoder_l2.inverse_transform(np.argmax(pred_l2, axis=1))
accuracy_l2 = np.mean(pred_labels_l2 == label_encoder_l2.inverse_transform(np.argmax(y_l2, axis=1)))

print(f'训练集 l2 的准确率: {accuracy_l2:.4f}')
print(f'Model_l2 time:{time_l2}')

490/490 [==============================] - 4s 9ms/step
训练集 l2 的准确率: 0.7473
Model_l2 time:71


In [ ]:
data = test

texts = data['text'].values
labels_l1 = data['l1'].values
labels_l2 = data['l2'].values

encoded_l1 = label_encoder_l1.transform(labels_l1)
encoded_l2 = label_encoder_l2.transform(labels_l2)
sequences = tokenizer.texts_to_sequences(texts)
X_padded = pad_sequences(sequences, maxlen=180)

y_l1 = to_categorical(encoded_l1)
y_l2 = to_categorical(encoded_l2)

pred_l1 = model_l1.predict(X_padded)
pred_l2 = model_l2.predict(X_padded)

pred_labels_l1 = label_encoder_l1.inverse_transform(np.argmax(pred_l1, axis=1))
pred_labels_l2 = label_encoder_l2.inverse_transform(np.argmax(pred_l2, axis=1))


accuracy_l1 = np.mean(pred_labels_l1 == labels_l1)
accuracy_l2 = np.mean(pred_labels_l2 == labels_l2)

print(f"Accuracy for test_l1: {accuracy_l1:.4f}")
print(f"Accuracy for test_l2: {accuracy_l2:.4f}")

490/490 [==============================] - 3s 6ms/step
Accuracy for test_l1: 0.9118
Accuracy for test_l2: 0.7473


In [ ]:
cor1 = pred_labels_l1 == labels_l1
acc1 =np.mean(cor1)
print(f"Consistency Accuracy for l1: {acc1:.4f}")

cor1_indices = np.where(cor1)[0]
pred_labels_l2_filtered = pred_labels_l2[cor1_indices]
lable_l2_filtered = labels_l2[cor1_indices]

cor_l2 = pred_labels_l2_filtered == lable_l2_filtered
acc2 = np.sum(cor_l2)/len(labels_l2)
print(f"Consistency Accuracy for l2: {acc2:.4f}")

Consistency Accuracy for l1: 0.9118
Consistency Accuracy for l2: 0.7170
